Importing requried libaries

In [0]:
import sys
sys.path.append('/Workspace/Users/saik84328@gmail.com')

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from SetUp.Config import bronze_schema, silver_schema, gold_schema
from pyspark.sql.types import *
from datetime import datetime
import uuid


In [0]:
start_time = datetime.now()

In [0]:
%run /Workspace/Users/saik84328@gmail.com/DataBricksLearning/AuditData

In [0]:
%run /Workspace/Users/saik84328@gmail.com/DataBricksLearning/ValidationFramework

Handling Schema

In [0]:


hospital_schema =F.StructType([
    StructField("Id", StringType(), True),
    StructField("Hosiptalname", StringType(), True),
    StructField("Address", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Zip", StringType(), True),
    StructField("Lattitude", DoubleType(), True),
    StructField("Longtitude", DoubleType(), True),
    StructField("Phone", IntegerType(), True),
    StructField("Revenue", DoubleType(), True),
    StructField("Utilization", IntegerType(), True)
])

Reading Source Data

In [0]:


df_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(hospital_schema)
    .load("/Volumes/helathcare_bronze/default/helathcare/organizations.csv")

    .withColumn("Readtimestamp", F.current_timestamp())

    .withColumn("Filename", F.col("_metadata.file_name"))

    .withColumn("file_size", F.col("_metadata.file_size"))
)

display(df_raw)

Creating Bronze Table

In [0]:

# Drop table if exists to avoid metadata mismatch
#spark.sql(f"DROP TABLE IF EXISTS helathcare_bronze.{bronze_schema}.PatientData")

df_raw.write.format("delta").option("delta.enableChangeDataFeed","true").mode("overwrite").saveAsTable(f"helathcare_bronze.{bronze_schema}.HosiptialData")

In [0]:
%sql
select * from  `helathcare_bronze`.`helathcare_bronze`.`HosiptialData`

Silver Validation


Checking Nulls

In [0]:
silver_df = spark.table("helathcare_bronze.helathcare_bronze.HosiptialData")

#need to replace null with 0 in phone number colum


In [0]:
validation_result = run_validations(
    silver_df,
    ["Id"]
)

print(f"Duplicate Count : {validation_result['duplicates']}")

print(
    f"Primary Key Status : "
    f"{validation_result['primary_key']['status']}"
)

print("\nColumns Having Null Values:")

display(validation_result["nulls"])

Handling columns wise Nulls 

In [0]:

# Column-wise replacement values
replace_dict = {
    "Phone": 9899981291
}

for col_name, replace_value in replace_dict.items():

    silver_df = silver_df.withColumn(
        col_name,

        F.when(
            F.col(col_name).isNull(),
            F.lit(replace_value)
        ).otherwise(F.col(col_name))
    )

display(silver_df)

Handling Duplicates

Standlize the data

%md
Standardized text by converting the first character of each record to uppercase.

In [0]:
silver_df = standardize_string_columns(silver_df)

In [0]:
silver_df = silver_df.withColumnRenamed(
    "Id",
    "Patient_id"
)
display(silver_df)

Selecting Requried Columns

In [0]:
silver_df = silver_df.select(
    "Patient_id",
    "Hosiptalname",
    "Address",
    "City",
    "State",
    "Zip",
    "Lattitude",
    "Longtitude",
    "Phone",
    "Revenue",
    "Utilization",
    "Filename"
)

In [0]:
validation_result = run_validations(
    silver_df,
    ["Patient_id"]
)

print(f"Duplicate Count : {validation_result['duplicates']}")

print(
    f"Primary Key Status : "
    f"{validation_result['primary_key']['status']}"
)

print("\nColumns Having Null Values:")

display(validation_result["nulls"])


#conclusion
# --in DRIVERS have null values,so fill with seuence so i taken min and max for that increase max valu by 1 --max S99999871,min-S99911728
# ---PREFIX have null based on gender column need to fill mr or mis in perfix
#---FIPS have nulls values , need to fill with county name having filps code
#

In [0]:
#create tempview
silver_df.createOrReplaceTempView(
    "source_Hosptial"
)


In [0]:
# silver_df.write.format("delta") \
#     .option("delta.enableChangeDataFeed", "true") \
#     .option("mergeSchema", "true") \
#     .mode("append") \
#     .saveAsTable(f"helathcare_silver.{silver_schema}.SL_Hosptial")

In [0]:
%sql
select count(*) from helathcare_silver.helathcare_silver.SL_Hosptial tgt

In [0]:
hosptial_count=silver_df.count()
print(hosptial_count)


In [0]:
%sql
MERGE INTO helathcare_silver.helathcare_silver.SL_Hosptial tgt

USING source_Hosptial src

ON tgt.patient_id = src.patient_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *

In [0]:
end_time = datetime.now()

duration_seconds = int(
    (end_time - start_time).total_seconds()
)

print(duration_seconds)

In [0]:
from pyspark.sql.types import LongType
from datetime import datetime

# Get Workflow Run ID
try:
    run_id = dbutils.jobs.taskContext().taskRunId()
except:
    run_id = f"MANUAL_{datetime.now().strftime('%Y%m%d%H%M%S')}"

target_table = "helathcare_silver.helathcare_silver.SL_Hosptial"

# Get metadata
notebook_name, table_name, layer = get_audit_metadata(target_table)

status = "SUCCESS"
error_message = None
record_count = 0

In [0]:
# Duplicate Check

duplicate_check_status = (
    "PASS"
    if validation_result["duplicates"] == 0
    else "FAIL"
)

# Primary Key Check

primary_key_status = (
    validation_result["primary_key"]["status"]
)

# Null Check

null_count = (
    validation_result["nulls"]
    .agg(F.sum("null_count"))
    .collect()[0][0]
)

null_check_status = (
    "PASS"
    if null_count == 0
    else "FAIL"
)

# Standardization

standardization_status = "PASS"

In [0]:
end_time = datetime.now()

write_audit(
    target_table=target_table,
    run_id=run_id,
    record_count=hosptial_count,
    start_time=start_time,
    end_time=end_time,
    status=status,
    duplicate_check_status=duplicate_check_status,
    primary_key_status=primary_key_status,
    null_check_status=null_check_status,
    standardization_status=standardization_status,
    error_message=error_message
)